In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))

In [2]:
from modules.lizard import create_lizard_attention_config


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Users/mahdikhashan/.conda/envs/xai_proj_space/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/mahdikhashan/.conda/envs/xai_proj_space/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/mahdikhashan/.conda/envs/xai_proj_space/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/mahdikhashan/.conda/envs/xai_proj_space/lib/python3.10/site-pa

In [3]:
import torch

In [32]:
cfg = create_lizard_attention_config(
    batch_size=2,
    seq_len=8192 * 2,
    hidden_size=256,
    num_heads=4,
    dtype="float32",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

In [33]:
from modules.lizard_attention_block_pytorch import LizardAttentionBlock

In [34]:
torch_dtype = cfg.dtype.to_torch()
head_dim = cfg.hidden_size // cfg.num_heads

print(f"Configuring model with: {cfg}")
print(f"Head Dimension: {head_dim}")

model = (
    LizardAttentionBlock(
        d_model=cfg.hidden_size, n_heads=cfg.num_heads, window_size=32, alpha=0.5, m=4
    )
    .to(cfg.device)
    .to(torch_dtype)
)

q = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)
k = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)
v = torch.randn(
    cfg.batch_size,
    cfg.num_heads,
    cfg.seq_len,
    head_dim,
    device=cfg.device,
    dtype=torch_dtype,
)

output = model(q, k, v)

print(f"Input shape:  {q.shape}")
print(f"Output shape: {output.shape}")

assert q.shape == output.shape

Configuring model with: LizardAttentionBlockConfig(batch_size=2, seq_len=16384, hidden_size=256, num_heads=4, dtype=DTypeConfig(name='float32'), device='cpu')
Head Dimension: 64
Input shape:  torch.Size([2, 4, 16384, 64])
Output shape: torch.Size([2, 4, 16384, 64])


In [35]:
def benchmark():
    model(q, k, v)

benchmark()

In [36]:
%timeit -r 5 -n 100 benchmark_step()

904 ms ± 49.8 ms per loop (mean ± std. dev. of 5 runs, 100 loops each)
